# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tuborg49/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [11]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Enter your Hugging Face READ token: ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

print("Hugging Face warehouse connection ready.")


Hugging Face warehouse connection ready.


One row represents one pseudonymized content item for a client on a report date. I will use the March 2026 month as the mid-panel working window.

Features: I will use five fields that are available before the prediction decision: sessions_organic, sessions_direct, scroll_events, client_has_gsc, and client_has_ga4.

Label / proxy: The provisional target is whether a content item is declining in the outcome window. I will define and verify this proxy from fields that are actually available in the warehouse before using it for modeling.

Context: content_hash_id identifies the content item, client_hash_id identifies the client, and report_date identifies the observation date.

Excluded: I will exclude fields that contain future outcome information or are derived from the target, because using them as features would cause data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# March 2026 data path
MAR = f"'{FACT}/month=2026-03/*.parquet'"

print(MAR)

'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'


In [14]:
# Query 1 — Verify the grain

grain_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id || '|' || client_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS distinct_content_client_dates,
    COUNT(DISTINCT content_hash_id || '|' || client_hash_id)
        AS distinct_content_client_pairs
FROM {MAR}
""").df()

grain_check

,total_rows,distinct_content_client_dates,distinct_content_client_pairs
0,9841378,9841378,331437


In [15]:
# Check the actual columns in the March 2026 data

columns_check = con.execute(f"""
DESCRIBE SELECT *
FROM {MAR}
""").df()

columns_check

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [16]:
# Query 2 — Verify row count and date window

window_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {MAR}
""").df()

window_check

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [17]:
# Query 3 — Check missing values

missing_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(content_hash_id) AS missing_content_hash_id,
    COUNT(*) - COUNT(client_hash_id) AS missing_client_hash_id,
    COUNT(*) - COUNT(report_date) AS missing_report_date,
    COUNT(*) - COUNT(sessions_organic) AS missing_sessions_organic,
    COUNT(*) - COUNT(sessions_direct) AS missing_sessions_direct,
    COUNT(*) - COUNT(scroll_events) AS missing_scroll_events
FROM {MAR}
""").df()

missing_check

,total_rows,missing_content_hash_id,missing_client_hash_id,missing_report_date,missing_sessions_organic,missing_sessions_direct,missing_scroll_events
0,9841378,0,0,0,3018741,3018741,3018741


In [18]:
# Five-feature frame for the March 2026 working slice

features = [
    "sessions_organic",
    "sessions_direct",
    "scroll_events",
    "client_has_gsc",
    "client_has_ga4"
]

feature_frame = con.execute(f"""
SELECT
    content_hash_id,
    client_hash_id,
    {", ".join(features)}
FROM {MAR}
""").df()

feature_frame.head()

,content_hash_id,client_hash_id,sessions_organic,sessions_direct,scroll_events,client_has_gsc,client_has_ga4
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,<NA>,<NA>,<NA>,True,False
1,content_05597932fe4da067,client_73cda7b4e4f265ea,<NA>,<NA>,<NA>,True,False
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,<NA>,<NA>,<NA>,True,False
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,<NA>,<NA>,<NA>,True,False
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,<NA>,<NA>,<NA>,True,False


### Five features and availability

1. `sessions_organic` — Knowable at the decision moment because historical organic sessions have already been recorded.

2. `sessions_direct` — Knowable at the decision moment because historical direct sessions have already been recorded.

3. `scroll_events` — Knowable at the decision moment because historical engagement events have already been collected.

4. `client_has_gsc` — Knowable at the decision moment because Search Console availability for the client is already known.

5. `client_has_ga4` — Knowable at the decision moment because Analytics availability for the client is already known.

In [19]:
# Inspect fields that may contain the trend/outcome information

[c for c in columns_check["column_name"].tolist()
 if any(word in c.lower() for word in ["trend", "declin", "change", "organic", "session"])]

['ga4_sessions',
 'ga4_engaged_sessions',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai']

In [20]:
# Leakage experiment
# We create a provisional future-decline label:
# 1 = next day's organic sessions are lower than today's.

leakage_check = con.execute(f"""
WITH daily AS (
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        sessions_organic,
        LEAD(sessions_organic) OVER (
            PARTITION BY content_hash_id, client_hash_id
            ORDER BY report_date
        ) AS next_day_sessions_organic
    FROM {MAR}
)

SELECT
    COUNT(*) AS rows_with_next_day,
    COUNT(*) FILTER (
        WHERE next_day_sessions_organic IS NOT NULL
    ) AS rows_with_future_value,
    COUNT(*) FILTER (
        WHERE next_day_sessions_organic < sessions_organic
    ) AS declining_rows
FROM daily
WHERE next_day_sessions_organic IS NOT NULL
""").df()

leakage_check

,rows_with_next_day,rows_with_future_value,declining_rows
0,6655313,6655313,142915


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4) Data limits

This data cannot fully tell us whether refreshing a page will improve its future performance because the observed history is not evenly balanced across all content and clients. Some early rows depend on GSC availability, and overlapping time windows can also affect the observations. The next-day decline outcome is a useful proxy, but it is not the same as measuring the actual impact of a content refresh.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 4) Limitation

A key limitation is that the March 2026 warehouse slice contains daily performance observations, while the final refresh decision requires a future outcome. The provisional declining label is therefore only a proxy and does not directly measure whether refreshing a page would improve its future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.